# 06 — SCR-Guided Target-Aware DifIISR

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shelly-serafimovich/target-aware-infrared-sr/blob/main/notebooks/06_target_aware_difiisr_v2.ipynb)

Zero-shot target-aware refinement of the official one-step DifIISR prediction. The frozen DifIISR UNet and autoencoder are not retrained. Candidate locations come only from the frozen LR image; GT boxes remain evaluation-only.

The final experiment in this notebook uses an explicit differentiable **SCR guidance objective** plus a fidelity penalty to the unguided DifIISR prediction.


## 1. Setup

This notebook is standalone as long as Notebook 05 has already created the saved pilot manifest on Drive. Run on a GPU runtime. It recreates the pinned DifIISR environment and reuses the frozen LR; you do **not** need Notebook 05 open in the same runtime.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
from pathlib import Path
import os,sys,subprocess,random
import numpy as np,pandas as pd,torch
from PIL import Image

PROJECT_REPO='https://github.com/shelly-serafimovich/target-aware-infrared-sr.git'
DIFIISR_REPO='https://github.com/zirui0625/DifIISR.git'
DIFIISR_COMMIT='09ca97ea48d481656dd8090e84099c963059ac41'
SEED=12345
DATASET_ROOT=Path('/content/drive/MyDrive/thermal-drone-dataset')
LR_DIR=DATASET_ROOT/'difiisr_baseline'/'valid'/'LR_difiisr_train_degradation_v2'
DYN_OUT=DATASET_ROOT/'difiisr_target_dynamics'
TA_OUT=DATASET_ROOT/'target_aware_difiisr'/'pilot_v1'; TA_OUT.mkdir(parents=True,exist_ok=True)
project_dir=Path('/content/target-aware-infrared-sr'); difiisr_dir=Path('/content/DifIISR')

def run(cmd,cwd=None):
    print('$',' '.join(map(str,cmd)))
    subprocess.run(list(map(str,cmd)),cwd=cwd,check=True)

if not project_dir.exists(): run(['git','clone',PROJECT_REPO,project_dir])
if not difiisr_dir.exists(): run(['git','clone',DIFIISR_REPO,difiisr_dir])
run(['git','fetch','--all'],cwd=difiisr_dir)
run(['git','checkout','--detach',DIFIISR_COMMIT],cwd=difiisr_dir)

ENV=Path('/content/difiisr-env'); PY=str(ENV/'bin'/'python')
if not Path(PY).exists():
    run([sys.executable,'-m','pip','install','-q','uv'])
    run([sys.executable,'-m','uv','venv','--python','3.10',ENV])
    run([sys.executable,'-m','uv','pip','install','--python',PY,'torch==2.1.1','torchvision==0.16.1','xformers==0.0.23','numpy==1.25.2','scipy==1.9.3','packaging==24.2','setuptools==69.5.1'])
    run([sys.executable,'-m','uv','pip','install','--python',PY,'pyiqa==0.1.12','basicsr==1.4.2','opencv-python','matplotlib','timm','pandas','pillow','scikit-learn','scikit-image','lpips','loguru','omegaconf','six','tqdm','albumentations','einops','imageio','gdown'])

weights=difiisr_dir/'weights'; weights.mkdir(exist_ok=True)
ckpt=weights/'DifIISR.pth'
if not ckpt.exists(): run([PY,'-m','gdown','1PhRvk1Dlp3CCrPkrxRfNbNZ3fNgviDVZ','-O',ckpt])
ae=weights/'autoencoder_vq_f4.pth'
if not ae.exists():
    run([PY,'-m','gdown','https://github.com/zsyOAOA/ResShift/releases/download/v2.0/autoencoder_vq_f4.pth','-O',ae])

manifest=DYN_OUT/'pilot_targets.csv'
assert manifest.exists(), f'Missing pilot manifest: {manifest}. Run Notebook 05 target selection once.'
print('Frozen LR:',LR_DIR)
print('Pilot manifest:',manifest)
print('Output:',TA_OUT)


## 2. Method — Tal-aligned SCR guidance

This implementation now follows the SCR construction used in Tal's notebook rather than the earlier simplified local-contrast surrogate.

- **Core:** 31×31 pixels (`N=961`).
- **Ring:** 61×61 neighborhood excluding the 31×31 core (`N=2760`).
- **SCR:** `(max(Core) - mean(Ring)) / max(std(Ring), sigma_floor)`.
- **Noise floor:** `sigma_floor = 2.247122` in 8-bit intensity units, calibrated as the 5th percentile of positive sky-ring standard deviations.
- **Soft gate:** `w = ReLU(SCR - 3.5)`.
- **Spatial aggregation:** stable LogSumExp with `beta = 2.0`.

The guidance is zero-shot and uses no GT boxes. The DifIISR UNet and autoencoder remain frozen. Hoyer is intentionally left out of this first final-method pilot so we can isolate whether explicit SCR guidance alone improves preservation/detection; it can be added as a separate ablation later.


## 3. Create the pilot script


In [ ]:
script=TA_OUT/'run_scr_guided_v1.py'
script.write_text(r'''import argparse,sys,random,time,gc
from pathlib import Path
import numpy as np,pandas as pd,torch
import torch.nn.functional as F
from PIL import Image
from omegaconf import OmegaConf

sys.path.insert(0,'/content/DifIISR')
import ldm.modules.diffusionmodules.model as ae_model
ae_model.XFORMERS_IS_AVAILBLE=False
from sampler import DifIISRSampler

ap=argparse.ArgumentParser()
ap.add_argument('--manifest'); ap.add_argument('--lr'); ap.add_argument('--out'); ap.add_argument('--config')
ap.add_argument('--seed',type=int,default=12345); ap.add_argument('--steps',type=int,default=4)
ap.add_argument('--lr-guidance',type=float,default=.08); ap.add_argument('--lambda-fid',type=float,default=.15)
a=ap.parse_args()

random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed); torch.cuda.manual_seed_all(a.seed)

# Exact DifIISR configuration used by the working pilot.
cfg=OmegaConf.load(a.config)
cfg.model.ckpt_path='/content/DifIISR/weights/DifIISR.pth'
cfg.diffusion.params.sf=4
cfg.autoencoder.ckpt_path='/content/DifIISR/weights/autoencoder_vq_f4.pth'

s=DifIISRSampler(cfg,chop_size=512,chop_stride=448,chop_bs=1,use_fp16=True,seed=a.seed,ddim=True)

df=pd.read_csv(a.manifest)
lrdir=Path(a.lr)
out=Path(a.out)
(out/'SR').mkdir(parents=True,exist_ok=True)
(out/'BASE').mkdir(parents=True,exist_ok=True)

def pad64(y):
 h,w=y.shape[-2:]; ph=(64-h%64)%64; pw=(64-w%64)%64
 return F.pad(y,(0,pw,0,ph),mode='reflect'),h,w

def candidate_mask(y):
 g=y.mean(1,keepdim=True)
 mu=F.avg_pool2d(g,7,1,3)
 pos=F.relu(g-mu)
 flat=pos.flatten(1)
 scale=torch.quantile(flat,.95,dim=1,keepdim=True).view(-1,1,1,1).clamp_min(1e-4)
 m=(pos/scale).clamp(0,1)
 return F.relu((m-.35)/.65)

def scr_map_tal(img):
 g=((img.mean(1,keepdim=True)+1.)*.5*255.).clamp(0,255)
 core_k,outer_k=31,61
 n_core=float(core_k*core_k); n_outer=float(outer_k*outer_k); n_ring=n_outer-n_core
 core_sum=F.avg_pool2d(g,core_k,1,core_k//2)*n_core
 core_sq=F.avg_pool2d(g*g,core_k,1,core_k//2)*n_core
 outer_sum=F.avg_pool2d(g,outer_k,1,outer_k//2)*n_outer
 outer_sq=F.avg_pool2d(g*g,outer_k,1,outer_k//2)*n_outer
 ring_sum=outer_sum-core_sum; ring_sq=outer_sq-core_sq
 ring_mu=ring_sum/n_ring
 ring_var=(ring_sq/n_ring-ring_mu*ring_mu).clamp_min(0.)
 ring_std=torch.sqrt(ring_var+1e-12)
 core_max=F.max_pool2d(g,core_k,1,core_k//2)
 sigma_floor=2.247122
 return (core_max-ring_mu)/torch.clamp(ring_std,min=sigma_floor)

def scr_objective(img,mask):
 scr=scr_map_tal(img)
 w=F.relu(scr-3.5)
 support=(mask>0).float()
 score=w*scr
 valid=score.masked_fill(support<=0,-1e4)
 beta=2.0
 flat=valid.flatten(1)
 active=(support.flatten(1).sum(1)>0)
 lse=torch.logsumexp(beta*flat,dim=1)/beta
 return torch.where(active,lse,torch.zeros_like(lse)).mean()

def save_gray(x,path,h,w):
 x=x[...,:h*4,:w*4]
 im=((x[0].float().mean(0).cpu().numpy()+1.)*.5*255.).clip(0,255).astype(np.uint8)
 Image.fromarray(im).save(path)

rows=[]
run_start=time.time()

for k,r in df.iterrows():
 img_start=time.time()
 ip=lrdir/r.image
 sr_path=out/'SR'/r.image
 base_path=out/'BASE'/r.image

 # Resume safety: only skip when the paired outputs both exist.
 if sr_path.exists() and base_path.exists():
  print(f'{k+1} / {len(df)} SKIP {r.image}',flush=True)
  continue

 arr=np.asarray(Image.open(ip).convert('RGB'),dtype=np.float32)/255.
 y=torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).cuda(); y=(y-.5)/.5
 y,h,w=pad64(y)
 mk_lr=candidate_mask(y)
 mk=F.interpolate(mk_lr,scale_factor=4,mode='bilinear',align_corners=False)
 model_kwargs={'lq':y} if s.configs.model.params.cond_lq else None

 # Official one-step DifIISR. BASE and guided SR share exactly the same z0.
 with torch.no_grad():
  z_y=s.base_diffusion.encode_first_stage(y,s.autoencoder,up_sample=True)
  noise=torch.randn_like(z_y)
  z=s.base_diffusion.prior_sample(z_y,noise)
  i=s.base_diffusion.num_timesteps-1
  t=torch.tensor([i],device=y.device)
  o=s.base_diffusion.ddim_sample(model=s.model,x=z,y=z_y,t=t,clip_denoised=False,model_kwargs=model_kwargs)
  z0=o['pred_xstart'].detach()
  base=s.base_diffusion.decode_first_stage(z0,s.autoencoder).detach().clamp(-1,1)
  scr0=float(scr_objective(base.float(),mk).detach())

 # Free diffusion-only tensors before the differentiable decoder passes.
 del z_y,noise,z,t,o
 gc.collect()
 torch.cuda.empty_cache()

 zg=z0.float().detach().requires_grad_(True)
 del z0
 opt=torch.optim.Adam([zg],lr=a.lr_guidance)
 hist=[]

 # Same frozen 4-step SCR guidance as the pilot.
 # Deliberately do NOT call gc.collect()/empty_cache() inside these steps.
 for q in range(a.steps):
  opt.zero_grad(set_to_none=True)
  dec=s.base_diffusion.decode_first_stage(zg,s.autoencoder,no_grad=False).float()
  scr=scr_objective(dec,mk)
  fid=F.mse_loss(dec,base.float())
  loss=-scr+a.lambda_fid*fid
  loss.backward()
  opt.step()
  hist.append((float(scr.detach()),float(fid.detach())))
  del dec,scr,fid,loss

 with torch.no_grad():
  pred=s.base_diffusion.decode_first_stage(zg,s.autoencoder).clamp(-1,1)

 # Save the paired unguided BASE and guided SR.
 save_gray(base,base_path,h,w)
 save_gray(pred,sr_path,h,w)

 rec=dict(image=r.image,scr_start=scr0,scr_end=hist[-1][0],
          delta_scr=hist[-1][0]-scr0,fidelity_end=hist[-1][1])
 rows.append(rec)

 # Append immediately so a Colab interruption does not lose the log.
 log_path=out/'run_log.csv'
 pd.DataFrame([rec]).to_csv(log_path,mode='a',header=not log_path.exists(),index=False)

 elapsed=time.time()-img_start
 done=k+1
 rate=60.0/elapsed if elapsed>0 else 0.0
 eta_min=(len(df)-done)*elapsed/60.0
 print(f'{done} / {len(df)} {r.image} | {elapsed:.1f}s/img | {rate:.2f} img/min | ETA {eta_min/60:.1f}h | SCR {scr0:.3f} -> {hist[-1][0]:.3f} | fid {hist[-1][1]:.6f}',flush=True)

 del y,mk,mk_lr,base,pred,zg,opt
 gc.collect()
 torch.cuda.empty_cache()

print(f'DONE {len(df)} | total {(time.time()-run_start)/3600:.2f}h',flush=True)
''')
print('Updated runner:',script)


## 4. Tal-aligned SCR-guided pilot — 45 images

Run the corrected SCR guidance first on the same 45-image pilot. This is the gate before full validation: we check that Tal's SCR objective increases while fidelity remains controlled, then evaluate the frozen YOLO before committing to all 2,115 images.


### 4.1 Run the 45-image v1 pilot

This cell intentionally uses the original v1 settings. The stronger 8-step / 0.40 experiment is not mixed into the frozen v1 validation experiment.


In [ ]:
manifest=DATASET_ROOT/'difiisr_target_dynamics'/'pilot_targets.csv'
SCR_PILOT_OUT=DATASET_ROOT/'target_aware_difiisr'/'scr_guided_pilot_v1'
SCR_PILOT_OUT.mkdir(parents=True,exist_ok=True)
cmd=[PY,'-u',str(script),'--manifest',str(manifest),'--lr',str(LR_DIR),'--out',str(SCR_PILOT_OUT),
     '--config','/content/DifIISR/configs/DifIISR_test.yaml','--seed','12345',
     '--steps','4','--lr-guidance','0.08','--lambda-fid','0.15']
env=os.environ.copy(); env['CUDA_VISIBLE_DEVICES']='0'; env['MPLBACKEND']='Agg'
result=subprocess.run(list(map(str,cmd)),cwd='/content/DifIISR',env=env,text=True)
if result.returncode!=0: raise RuntimeError('SCR-guided pilot failed.')
log=pd.read_csv(SCR_PILOT_OUT/'run_log.csv')
display(log[['scr_start','scr_end','delta_scr','fidelity_end']].describe())
print('Images with SCR increase:',int((log.delta_scr>0).sum()),'/',len(log))


## 5. Sanity check before YOLO

First compare a few Original DifIISR and Target-Aware outputs visually and verify that the intervention is local/subtle rather than creating bright artifacts. If this passes, the next notebook step is the frozen-YOLO comparison on the pilot, then validation-scale ablation.


In [ ]:
import matplotlib.pyplot as plt
ORIG=DATASET_ROOT/'difiisr_baseline'/'valid'/'SR_difiisr_train_degradation_v2'
names=pd.read_csv(manifest).image.drop_duplicates().head(5).tolist()
for name in names:
 a=np.asarray(Image.open(ORIG/name).convert('L'))
 b=np.asarray(Image.open(SCR_PILOT_OUT/'SR'/name).convert('L'))
 fig,ax=plt.subplots(figsize=(8,4))
 ax.imshow(np.concatenate([a,b],axis=1),cmap='gray')
 ax.set_title('Original DifIISR  |  SCR-guided v1')
 ax.axis('off'); plt.show()


## 5.1 Frozen YOLO — GT-matched pilot check

Before full validation, compare Original DifIISR and SCR-guided v1 on the same 45 annotated pilot targets with the frozen detector. A detection counts as GT-matched when the best prediction has IoU ≥ 0.25. This cell does not retrain or modify YOLO.


In [ ]:
from ultralytics import YOLO

YOLO_WEIGHTS=DATASET_ROOT/'detector'/'weights'/'thermal_drone_yolo11n_corrected_v1_best.pt'
TA_SR=SCR_PILOT_OUT/'SR'
pilot=pd.read_csv(manifest).copy()
assert len(pilot)==45 and pilot.image.nunique()==45
model=YOLO(str(YOLO_WEIGHTS))

def iou_xyxy(a,b):
    x1=max(a[0],b[0]); y1=max(a[1],b[1]); x2=min(a[2],b[2]); y2=min(a[3],b[3])
    inter=max(0.,x2-x1)*max(0.,y2-y1)
    aa=max(0.,a[2]-a[0])*max(0.,a[3]-a[1]); bb=max(0.,b[2]-b[0])*max(0.,b[3]-b[1])
    return inter/(aa+bb-inter+1e-12)

def eval_folder(folder,label):
    rows=[]
    for _,r in pilot.iterrows():
        p=folder/r.image
        with Image.open(p) as im: W,H=im.size
        gt=[(r.xc-r.bw/2)*W,(r.yc-r.bh/2)*H,(r.xc+r.bw/2)*W,(r.yc+r.bh/2)*H]
        pred=model.predict(str(p),conf=.001,iou=.7,verbose=False)[0]
        best_iou=0.; best_conf=0.
        if pred.boxes is not None and len(pred.boxes):
            boxes=pred.boxes.xyxy.detach().cpu().numpy(); confs=pred.boxes.conf.detach().cpu().numpy()
            vals=[iou_xyxy(gt,b) for b in boxes]
            j=int(np.argmax(vals)); best_iou=float(vals[j])
            if best_iou>=.25: best_conf=float(confs[j])
        rows.append(dict(image=r.image,size_bin=r.lr_size_bin,method=label,best_iou=best_iou,detected=best_iou>=.25,gt_conf=best_conf))
    return pd.DataFrame(rows)

orig_y=eval_folder(ORIG,'Original DifIISR')
ta_y=eval_folder(TA_SR,'SCR-guided v1')
cmp=orig_y.merge(ta_y,on=['image','size_bin'],suffixes=('_orig','_scr'))
print('Targets:',len(cmp))
print('Original recall:',int(cmp.detected_orig.sum()),'/',len(cmp),'=',cmp.detected_orig.mean())
print('SCR-guided recall:',int(cmp.detected_scr.sum()),'/',len(cmp),'=',cmp.detected_scr.mean())
print('Saved by SCR guidance:',int((~cmp.detected_orig & cmp.detected_scr).sum()))
print('Lost by SCR guidance:',int((cmp.detected_orig & ~cmp.detected_scr).sum()))
both=cmp[cmp.detected_orig & cmp.detected_scr].copy()
if len(both):
    both['delta_conf']=both.gt_conf_scr-both.gt_conf_orig
    print('Detected by both:',len(both))
    print('Mean GT confidence — Original:',both.gt_conf_orig.mean())
    print('Mean GT confidence — SCR-guided:',both.gt_conf_scr.mean())
    print('Mean Δ confidence:',both.delta_conf.mean())
    print('Median Δ confidence:',both.delta_conf.median())
print('\nRecall by LR target-size bin:')
display(cmp.groupby('size_bin',observed=True).agg(N=('image','size'),original_recall=('detected_orig','mean'),scr_recall=('detected_scr','mean')))
YOLO_PILOT_CSV=SCR_PILOT_OUT/'yolo_gt_matched_comparison.csv'
cmp.to_csv(YOLO_PILOT_CSV,index=False)
print('Saved:',YOLO_PILOT_CSV)


## 6. Full validation — Tal-aligned SCR-guided v1

**Do not run this until the corrected 45-image pilot has been inspected.** After the pilot passes, freeze the settings and run the same SCR-guided method on all 2,115 validation images, including 297 background-only images. The test split remains untouched.


In [ ]:
# Build a resume-safe manifest from every frozen validation LR image.
# The known 480x270 LR outlier is deferred because vanilla autoencoder
# self-attention OOMs on that spatial size. It will be handled separately
# after the standard-size validation run completes.

FULL_OUT=DATASET_ROOT/'target_aware_difiisr'/'scr_guided_full_validation_v1'
FULL_SR=FULL_OUT/'SR'
FULL_BASE=FULL_OUT/'BASE'
FULL_OUT.mkdir(parents=True,exist_ok=True)
FULL_SR.mkdir(parents=True,exist_ok=True)
FULL_BASE.mkdir(parents=True,exist_ok=True)

assert LR_DIR.exists(), f'Missing frozen LR directory: {LR_DIR}'

exts={'.png','.jpg','.jpeg'}
all_images=sorted(p.name for p in LR_DIR.iterdir() if p.suffix.lower() in exts)
assert len(all_images)==2115, f'Expected 2115 validation images, found {len(all_images)}'

# Defer only images whose frozen LR size is not the standard 160x128.
from PIL import Image

deferred=[]
eligible=[]

for name in all_images:
    with Image.open(LR_DIR/name) as im:
        if im.size in {(160,128),(160,160)}:
            eligible.append(name)
        else:
            deferred.append((name,im.size))

# Resume safety: count an image complete only when BOTH paired outputs exist.
completed={
    name for name in eligible
    if (FULL_SR/name).exists() and (FULL_BASE/name).exists()
}

pending=[name for name in eligible if name not in completed]

PENDING_MANIFEST=FULL_OUT/'pending_manifest.csv'
pd.DataFrame({'image':pending}).to_csv(PENDING_MANIFEST,index=False)

DEFERRED_MANIFEST=FULL_OUT/'deferred_nonstandard_size.csv'
pd.DataFrame(
    [{'image':name,'width':size[0],'height':size[1]} for name,size in deferred]
).to_csv(DEFERRED_MANIFEST,index=False)

print('Total validation images:',len(all_images))
print('Standard-size eligible:',len(eligible))
print('Already completed (paired BASE+SR):',len(completed))
print('Still pending standard-size:',len(pending))
print('Deferred non-standard images:',len(deferred))

for name,size in deferred:
    print('  DEFERRED:',name,'size=',size)

print('Pending manifest:',PENDING_MANIFEST)
print('Deferred manifest:',DEFERRED_MANIFEST)


### 6.1 Run / resume the full validation

If Colab disconnects, rerun Setup, Section 3 (to recreate the script), and then Section 6. The manifest is rebuilt from the files already saved on Drive, so completed images are skipped.


In [ ]:
if len(pending)==0:
    print('FULL VALIDATION ALREADY COMPLETE ✅')
else:
    cmd=[PY,'-u',str(script),'--manifest',str(PENDING_MANIFEST),'--lr',str(LR_DIR),'--out',str(FULL_OUT),
         '--config','/content/DifIISR/configs/DifIISR_test.yaml','--seed','12345',
         '--steps','4','--lr-guidance','0.08','--lambda-fid','0.15']
    env=os.environ.copy(); env['CUDA_VISIBLE_DEVICES']='0'; env['MPLBACKEND']='Agg'
    print(f'Running frozen SCR-guided v1 on {len(pending)} pending images')
    print('steps=4 | guidance_lr=0.08 | lambda_fid=0.15 | seed=12345')
    result=subprocess.run(list(map(str,cmd)),cwd='/content/DifIISR',env=env,text=True)
    if result.returncode!=0:
        raise RuntimeError('Run stopped. Keep the Drive outputs and rerun Sections 1, 3, and 6 to resume.')


### 6.2 Completion check


In [ ]:
done=sorted(p.name for p in FULL_SR.iterdir() if p.suffix.lower() in exts)
print('Target-Aware SR images:',len(done),'/ 2115')
if len(done)==2115:
    print('FULL VALIDATION COMPLETE ✅')
else:
    print('Not finished yet — rerun Section 6 to resume.')


## 7. Full-validation evaluation plan

Once all 2,115 Target-Aware outputs exist, compare **Original DifIISR vs SCR-guided v1** on the same frozen validation split. Reuse the exact metric definitions already established in the earlier notebooks rather than redefining them here.

- Image-quality metrics: PSNR, SSIM, LPIPS, CLIP-IQA, MUSIQ, NIQE.
- Target metrics: SCR and calibrated Hoyer, plus target-vs-same-image-background AUROC.
- Frozen detector: GT-matched recall/confidence and detector metrics, stratified by LR target size; use the 297 background-only images for false-alarm analysis.
- Size strata: ≤2, 2–4, 4–8, 8–16, >16 LR pixels, plus background.

Do not touch the test split until the method and validation analysis are frozen.
